In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # E7.1 · Ingesta RAW → Bronze
# MAGIC
# MAGIC **M6 Big Data:** procesamiento batch  
# MAGIC **M7 Integración:** ingestión incremental con Auto Loader
# MAGIC
# MAGIC ## Principio de la capa Bronze
# MAGIC
# MAGIC La capa Bronze mantiene **fidelidad al origen**:
# MAGIC
# MAGIC - No limpia datos.
# MAGIC - No filtra registros.
# MAGIC - No deduplica.
# MAGIC - No explota estructuras anidadas.
# MAGIC - No aplica reglas de negocio.
# MAGIC
# MAGIC Sí incorpora metadata técnica para:
# MAGIC
# MAGIC - Auditoría.
# MAGIC - Linaje.
# MAGIC - Trazabilidad.
# MAGIC - Identificación de ejecución.
# MAGIC - Control de integridad.
# MAGIC - Control de evolución del schema.
# MAGIC
# MAGIC ## Modalidades
# MAGIC
# MAGIC **Batch**
# MAGIC - business
# MAGIC - user
# MAGIC - tip
# MAGIC - checkin
# MAGIC
# MAGIC **Incremental / Auto Loader**
# MAGIC - review
# MAGIC
# MAGIC > **Trazabilidad:** este notebook implementa los SBB de ingesta
# MAGIC > y almacenamiento definidos en la Arquitectura de Referencia
# MAGIC > del proyecto YelpConnect.


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F

import uuid
import hashlib

from datetime import (
    datetime,
    timezone
)


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN GENERAL
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "yelp_bronze"


# ------------------------------------------------------------
# Volume RAW
# ------------------------------------------------------------

RAW = (
    "/Volumes/"
    "workspace/"
    "yelp_bronze/"
    "raw"
)


# ------------------------------------------------------------
# Entidades procesadas por Batch
# ------------------------------------------------------------

ENTIDADES_BATCH = [
    "business",
    "user",
    "tip",
    "checkin"
]


# ------------------------------------------------------------
# Entidad procesada mediante Auto Loader
# ------------------------------------------------------------

ENTIDAD_STREAM = "review"


# ------------------------------------------------------------
# Metadata común
# ------------------------------------------------------------

SOURCE_SYSTEM = "YELP"

LAYER = "bronze"

PIPELINE_NAME = (
    "E7_01_raw_to_bronze"
)


# ------------------------------------------------------------
# Identificador único de ejecución
# ------------------------------------------------------------

execution_id = str(
    uuid.uuid4()
)


execution_started_at = (
    datetime
    .now(timezone.utc)
    .isoformat()
)


print("=" * 70)
print(
    "YELPCONNECT - PIPELINE RAW -> BRONZE"
)
print("=" * 70)

print(
    f"Catalog       : {CATALOG}"
)

print(
    f"Schema Bronze : {BRONZE_SCHEMA}"
)

print(
    f"RAW Volume    : {RAW}"
)

print(
    f"Pipeline      : {PIPELINE_NAME}"
)

print(
    f"Execution ID  : {execution_id}"
)

print(
    f"Inicio UTC    : {execution_started_at}"
)

print("=" * 70)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Validación del Schema Bronze
# MAGIC
# MAGIC El schema debería ser creado originalmente desde `00_setup`.
# MAGIC
# MAGIC Se mantiene `CREATE SCHEMA IF NOT EXISTS` para que el notebook
# MAGIC sea tolerante a una primera ejecución del entorno.


# COMMAND ----------

# ============================================================
# 3. CREAR / VALIDAR SCHEMA BRONZE
# ============================================================

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{BRONZE_SCHEMA}
    """
)


print(
    f"✔ Schema disponible: "
    f"{CATALOG}.{BRONZE_SCHEMA}"
)


# COMMAND ----------

# ============================================================
# 4. FUNCIÓN PARA AGREGAR METADATA BRONZE
# ============================================================

def add_bronze_metadata(
    df,
    entity,
    execution_id,
    source_file_expression
):
    """
    Agrega metadata técnica a la capa Bronze.

    IMPORTANTE:
    El checksum se calcula únicamente utilizando
    las columnas originales provenientes de RAW.

    No se incluyen:
      - execution_id
      - timestamp de ingesta
      - pipeline
      - layer

    De esta manera, el mismo registro fuente
    produce el mismo checksum en ejecuciones diferentes.
    """

    # --------------------------------------------------------
    # Columnas originales
    # --------------------------------------------------------

    source_columns = sorted(
        df.columns
    )


    # --------------------------------------------------------
    # Fingerprint del schema
    # --------------------------------------------------------

    schema_json = (
        df
        .schema
        .json()
    )


    schema_hash = (
        hashlib
        .sha256(
            schema_json.encode(
                "utf-8"
            )
        )
        .hexdigest()
    )


    # --------------------------------------------------------
    # Representación determinística del registro
    # --------------------------------------------------------

    record_json = F.to_json(
        F.struct(
            *[
                F.col(c)
                for c
                in source_columns
            ]
        ),
        options={
            "ignoreNullFields":
            "false"
        }
    )


    # --------------------------------------------------------
    # Metadata técnica
    # --------------------------------------------------------

    df_bronze = (
        df

        # Archivo RAW que originó el registro
        .withColumn(
            "_archivo_origen",
            source_file_expression
        )

        # Momento de ingestión
        .withColumn(
            "_fecha_ingesta",
            F.current_timestamp()
        )

        # Sistema origen
        .withColumn(
            "_source_system",
            F.lit(
                SOURCE_SYSTEM
            )
        )

        # Entidad origen
        .withColumn(
            "_source_entity",
            F.lit(
                entity
            )
        )

        # Pipeline
        .withColumn(
            "_pipeline_name",
            F.lit(
                PIPELINE_NAME
            )
        )

        # Ejecución
        .withColumn(
            "_execution_id",
            F.lit(
                execution_id
            )
        )

        # Capa
        .withColumn(
            "_layer",
            F.lit(
                LAYER
            )
        )

        # Fingerprint de la estructura
        .withColumn(
            "_schema_hash",
            F.lit(
                schema_hash
            )
        )

        # Fingerprint del contenido
        .withColumn(
            "_record_checksum",
            F.sha2(
                record_json,
                256
            )
        )
    )


    return df_bronze


# COMMAND ----------

# MAGIC %md
# MAGIC ---
# MAGIC # 5. Ingesta Batch
# MAGIC
# MAGIC Entidades:
# MAGIC
# MAGIC - Business
# MAGIC - User
# MAGIC - Tip
# MAGIC - Check-in
# MAGIC
# MAGIC Se realiza carga completa debido a que son datasets estáticos
# MAGIC o de volumen manejable dentro del alcance del Capstone.


# COMMAND ----------

# ============================================================
# 5. INGESTA BATCH
# ============================================================

batch_results = []


for entity in ENTIDADES_BATCH:

    print()
    print("=" * 70)

    print(
        f"PROCESANDO ENTIDAD: "
        f"{entity.upper()}"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # Archivo RAW
    # --------------------------------------------------------

    source_path = (
        f"{RAW}/"
        f"{entity}.json.gz"
    )


    # --------------------------------------------------------
    # Tabla destino
    # --------------------------------------------------------

    target_table = (
        f"{CATALOG}."
        f"{BRONZE_SCHEMA}."
        f"brz_{entity}"
    )


    print(
        f"Origen  : "
        f"{source_path}"
    )

    print(
        f"Destino : "
        f"{target_table}"
    )


    # --------------------------------------------------------
    # 5.1 Leer RAW
    # --------------------------------------------------------

    try:

        df_raw = (
            spark
            .read

            .format(
                "json"
            )

            .option(
                "mode",
                "PERMISSIVE"
            )

            .option(
                "columnNameOfCorruptRecord",
                "_corrupt_record"
            )

            .load(
                source_path
            )
        )

    except Exception as e:

        print(
            f"❌ ERROR leyendo "
            f"{entity}"
        )

        print(
            str(e)
        )

        raise


    # --------------------------------------------------------
    # 5.2 Validaciones mínimas
    # --------------------------------------------------------

    raw_count = (
        df_raw
        .count()
    )


    column_count = len(
        df_raw.columns
    )


    if raw_count == 0:

        raise Exception(
            f"{entity}: "
            "archivo RAW vacío."
        )


    if column_count == 0:

        raise Exception(
            f"{entity}: "
            "no se detectaron columnas."
        )


    print(
        f"RAW registros : "
        f"{raw_count:,}"
    )

    print(
        f"RAW columnas  : "
        f"{column_count}"
    )


    # --------------------------------------------------------
    # 5.3 Registros corruptos
    # --------------------------------------------------------

    corrupt_count = 0


    if (
        "_corrupt_record"
        in df_raw.columns
    ):

        corrupt_count = (
            df_raw

            .filter(
                F.col(
                    "_corrupt_record"
                ).isNotNull()
            )

            .count()
        )


    print(
        f"RAW corruptos : "
        f"{corrupt_count:,}"
    )


    # --------------------------------------------------------
    # 5.4 Agregar metadata
    # --------------------------------------------------------

    df_bronze = (
        add_bronze_metadata(
            df=df_raw,
            entity=entity,
            execution_id=(
                execution_id
            ),
            source_file_expression=(
                F.col("_metadata.file_path")
            )
        )
    )


    # --------------------------------------------------------
    # 5.5 Escritura Delta
    # --------------------------------------------------------

    try:

        (
            df_bronze

            .write

            .format(
                "delta"
            )

            # Bootstrap académico:
            # recreamos la tabla batch completa.
            .mode(
                "overwrite"
            )

            .option(
                "overwriteSchema",
                "true"
            )

            .saveAsTable(
                target_table
            )
        )

    except Exception as e:

        print(
            f"❌ ERROR escribiendo "
            f"{target_table}"
        )

        print(
            str(e)
        )

        raise


    # --------------------------------------------------------
    # 5.6 Validación post-carga
    # --------------------------------------------------------

    bronze_count = (
        spark
        .table(
            target_table
        )
        .count()
    )


    if (
        raw_count
        != bronze_count
    ):

        raise Exception(
            f"""
            ERROR DE RECONCILIACIÓN

            Entidad : {entity}

            RAW     : {raw_count}
            BRONZE  : {bronze_count}
            """
        )


    # --------------------------------------------------------
    # 5.7 Registrar resultado
    # --------------------------------------------------------

    batch_results.append(
        (
            entity,
            raw_count,
            bronze_count,
            corrupt_count
        )
    )


    print(
        f"✔ brz_{entity}: "
        f"{bronze_count:,} filas"
    )


# COMMAND ----------

# ============================================================
# 6. RESUMEN DE CARGA BATCH
# ============================================================

batch_summary_df = (
    spark
    .createDataFrame(
        batch_results,
        [
            "entity",
            "raw_records",
            "bronze_records",
            "corrupt_records"
        ]
    )
)


display(
    batch_summary_df
)


# COMMAND ----------

# MAGIC %md
# MAGIC ---
# MAGIC # 7. Ingesta incremental de Review
# MAGIC
# MAGIC `review` se procesa mediante **Databricks Auto Loader** porque:
# MAGIC
# MAGIC - Es la entidad principal del caso Trust & Safety.
# MAGIC - Tiene el mayor volumen.
# MAGIC - Permite demostrar ingesta incremental.
# MAGIC - Permite demostrar checkpoint.
# MAGIC - Permite demostrar schema tracking.
# MAGIC
# MAGIC El pipeline utiliza `availableNow=True`, por lo que procesa
# MAGIC todos los archivos pendientes y finaliza.


# COMMAND ----------

# ============================================================
# 7. CONFIGURACIÓN AUTO LOADER
# ============================================================

REVIEW_SOURCE = (
    f"{RAW}/"
    "review*.json.gz"
)


CHECKPOINT_BASE = (
    f"{RAW}/"
    "_checkpoints/"
    "review"
)


SCHEMA_LOCATION = (
    f"{CHECKPOINT_BASE}/"
    "schema"
)


CHECKPOINT_LOCATION = (
    f"{CHECKPOINT_BASE}/"
    "state"
)


REVIEW_TABLE = (
    f"{CATALOG}."
    f"{BRONZE_SCHEMA}."
    "brz_review"
)


print("=" * 70)
print(
    "AUTO LOADER - REVIEW"
)
print("=" * 70)

print(
    f"Source          : "
    f"{REVIEW_SOURCE}"
)

print(
    f"Schema Location : "
    f"{SCHEMA_LOCATION}"
)

print(
    f"Checkpoint      : "
    f"{CHECKPOINT_LOCATION}"
)

print(
    f"Destino         : "
    f"{REVIEW_TABLE}"
)

print("=" * 70)


# COMMAND ----------

# ============================================================
# 8. CREAR STREAM AUTO LOADER
# ============================================================

df_review_raw = (
    spark

    .readStream

    .format(
        "cloudFiles"
    )

    .option(
        "cloudFiles.format",
        "json"
    )

    .option(
        "cloudFiles.schemaLocation",
        SCHEMA_LOCATION
    )

    .option(
        "cloudFiles.schemaEvolutionMode",
        "addNewColumns"
    )

    .load(
        REVIEW_SOURCE
    )
)


# COMMAND ----------

# ============================================================
# 9. AGREGAR METADATA A REVIEW
# ============================================================

df_review_bronze = (
    add_bronze_metadata(
        df=df_review_raw,

        entity="review",

        execution_id=(
            execution_id
        ),

        source_file_expression=(
            F.col(
                "_metadata.file_path"
            )
        )
    )
)


# COMMAND ----------

# ============================================================
# 10. ESCRIBIR REVIEW MEDIANTE AUTO LOADER
# ============================================================

query = (
    df_review_bronze

    .writeStream

    .format(
        "delta"
    )

    .option(
        "checkpointLocation",
        CHECKPOINT_LOCATION
    )

    # Procesa todos los archivos
    # disponibles y termina.
    .trigger(
        availableNow=True
    )

    .toTable(
        REVIEW_TABLE
    )
)


query.awaitTermination()


print(
    "✔ Auto Loader finalizado."
)


# COMMAND ----------

# ============================================================
# 11. VALIDAR REVIEW
# ============================================================

review_count = (
    spark

    .table(
        REVIEW_TABLE
    )

    .count()
)


print(
    f"✔ brz_review: "
    f"{review_count:,} filas"
)


# COMMAND ----------

# MAGIC %md
# MAGIC ---
# MAGIC # 12. Validación general Bronze
# MAGIC
# MAGIC Se verifica la existencia y cantidad de registros de las
# MAGIC cinco entidades Bronze.


# COMMAND ----------

# ============================================================
# 12. VALIDAR LAS CINCO TABLAS
# ============================================================

BRONZE_TABLES = [
    "brz_business",
    "brz_user",
    "brz_tip",
    "brz_checkin",
    "brz_review"
]


validation_results = []


for table_name in BRONZE_TABLES:

    full_table = (
        f"{CATALOG}."
        f"{BRONZE_SCHEMA}."
        f"{table_name}"
    )


    total_records = (
        spark

        .table(
            full_table
        )

        .count()
    )


    validation_results.append(
        (
            table_name,
            total_records
        )
    )


validation_df = (
    spark
    .createDataFrame(
        validation_results,
        [
            "table_name",
            "total_records"
        ]
    )
)


display(
    validation_df
)


# COMMAND ----------

# ============================================================
# 13. VALIDAR METADATA BRONZE
# ============================================================

METADATA_COLUMNS = [
    "_archivo_origen",
    "_fecha_ingesta",
    "_source_system",
    "_source_entity",
    "_pipeline_name",
    "_execution_id",
    "_layer",
    "_schema_hash",
    "_record_checksum"
]


metadata_results = []


for table_name in BRONZE_TABLES:

    full_table = (
        f"{CATALOG}."
        f"{BRONZE_SCHEMA}."
        f"{table_name}"
    )


    df_table = (
        spark
        .table(
            full_table
        )
    )


    missing_columns = [
        c
        for c
        in METADATA_COLUMNS
        if c
        not in df_table.columns
    ]


    metadata_results.append(
        (
            table_name,
            len(
                missing_columns
            ) == 0,
            ", ".join(
                missing_columns
            )
        )
    )


metadata_validation_df = (
    spark
    .createDataFrame(
        metadata_results,
        [
            "table_name",
            "metadata_ok",
            "missing_columns"
        ]
    )
)


display(
    metadata_validation_df
)


# COMMAND ----------

# MAGIC %md
# MAGIC ---
# MAGIC # 14. Evidencia de linaje
# MAGIC
# MAGIC Para una muestra de registros de `brz_business`
# MAGIC se visualiza:
# MAGIC
# MAGIC - archivo origen;
# MAGIC - timestamp de ingesta;
# MAGIC - ejecución;
# MAGIC - sistema origen;
# MAGIC - checksum;
# MAGIC - fingerprint del schema.


# COMMAND ----------

# ============================================================
# 14. MUESTRA DE LINAJE
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            business_id,

            name,

            _archivo_origen,

            _fecha_ingesta,

            _source_system,

            _source_entity,

            _pipeline_name,

            _execution_id,

            _layer,

            _record_checksum,

            _schema_hash

        FROM
            {CATALOG}.
            {BRONZE_SCHEMA}.
            brz_business

        LIMIT 20
        """
    )
)


# COMMAND ----------

# ============================================================
# 15. RESUMEN DE EJECUCIÓN BUSINESS
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            _execution_id,

            _source_system,

            _source_entity,

            _pipeline_name,

            _layer,

            COUNT(*) AS total_records,

            COUNT(
                DISTINCT
                _record_checksum
            ) AS distinct_checksums,

            MIN(
                _fecha_ingesta
            ) AS ingestion_start,

            MAX(
                _fecha_ingesta
            ) AS ingestion_end

        FROM
            {CATALOG}.
            {BRONZE_SCHEMA}.
            brz_business

        GROUP BY

            _execution_id,

            _source_system,

            _source_entity,

            _pipeline_name,

            _layer

        ORDER BY
            ingestion_start DESC
        """
    )
)


# COMMAND ----------

# ============================================================
# 16. PERFIL BÁSICO BUSINESS
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            COUNT(*) AS total_records,

            COUNT(
                business_id
            ) AS business_id_not_null,

            COUNT(
                DISTINCT
                business_id
            ) AS unique_business_ids,

            SUM(
                CASE
                    WHEN business_id IS NULL
                    THEN 1
                    ELSE 0
                END
            ) AS business_id_nulls,

            COUNT(
                DISTINCT
                _record_checksum
            ) AS unique_checksums

        FROM
            {CATALOG}.
            {BRONZE_SCHEMA}.
            brz_business
        """
    )
)


# COMMAND ----------

# ============================================================
# 17. DELTA DETAIL
# ============================================================

display(
    spark.sql(
        f"""
        DESCRIBE DETAIL
        {CATALOG}.
        {BRONZE_SCHEMA}.
        brz_business
        """
    )
)


# COMMAND ----------

# ============================================================
# 18. DELTA HISTORY
# ============================================================

display(
    spark.sql(
        f"""
        DESCRIBE HISTORY
        {CATALOG}.
        {BRONZE_SCHEMA}.
        brz_business
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ---
# MAGIC # 19. Experimento obligatorio Auto Loader
# MAGIC
# MAGIC ## Ejecución 1
# MAGIC
# MAGIC Ejecutar el notebook con:
# MAGIC
# MAGIC `review.json.gz`
# MAGIC
# MAGIC Registrar la cantidad de filas de:
# MAGIC
# MAGIC `workspace.yelp_bronze.brz_review`
# MAGIC
# MAGIC ---
# MAGIC
# MAGIC ## Ejecución 2
# MAGIC
# MAGIC Ejecutar nuevamente únicamente las celdas de Auto Loader
# MAGIC **sin cargar archivos nuevos**.
# MAGIC
# MAGIC Resultado esperado:
# MAGIC
# MAGIC - `review.json.gz` no vuelve a procesarse.
# MAGIC - El número total de filas permanece igual.
# MAGIC
# MAGIC Esto ocurre porque el checkpoint conserva
# MAGIC el estado de los archivos procesados.
# MAGIC
# MAGIC ---
# MAGIC
# MAGIC ## Ejecución 3
# MAGIC
# MAGIC Agregar un archivo:
# MAGIC
# MAGIC `review_2.json.gz`
# MAGIC
# MAGIC y volver a ejecutar Auto Loader.
# MAGIC
# MAGIC Resultado esperado:
# MAGIC
# MAGIC - se procesa únicamente `review_2.json.gz`;
# MAGIC - los registros anteriores no se reprocesan.
# MAGIC
# MAGIC > El checkpoint persiste el estado de procesamiento,
# MAGIC > mientras que `schemaLocation` mantiene el seguimiento
# MAGIC > de la estructura detectada.


# COMMAND ----------

# MAGIC %md
# MAGIC # 20. Decisión Batch vs Incremental
# MAGIC
# MAGIC | Entidad | Modalidad | Justificación |
# MAGIC |---|---|---|
# MAGIC | **Business** | Batch | Dataset maestro, volumen manejable y menor necesidad de ingestión continua. |
# MAGIC | **User** | Batch | Dataset de referencia utilizado para relaciones y análisis del comportamiento. |
# MAGIC | **Tip** | Batch | Volumen manejable y uso secundario dentro del alcance actual. |
# MAGIC | **Check-in** | Batch | Archivo agregado por Business; su descomposición temporal será realizada posteriormente en Silver. |
# MAGIC | **Review** | Auto Loader | Entidad principal de Trust & Safety y de mayor volumen. Permite demostrar ingesta incremental mediante schema tracking y checkpoint. |


# COMMAND ----------

# MAGIC %md
# MAGIC # 21. Metadata técnica Bronze
# MAGIC
# MAGIC Todas las tablas incorporan:
# MAGIC
# MAGIC | Campo | Propósito |
# MAGIC |---|---|
# MAGIC | `_archivo_origen` | Archivo RAW que originó el registro |
# MAGIC | `_fecha_ingesta` | Momento de ingestión |
# MAGIC | `_source_system` | Sistema origen: YELP |
# MAGIC | `_source_entity` | Entidad de origen |
# MAGIC | `_pipeline_name` | Pipeline responsable |
# MAGIC | `_execution_id` | Identificador único de ejecución |
# MAGIC | `_layer` | Capa Bronze |
# MAGIC | `_schema_hash` | Fingerprint SHA-256 de la estructura fuente |
# MAGIC | `_record_checksum` | SHA-256 del contenido original del registro |
# MAGIC
# MAGIC ### Diferencia entre controles
# MAGIC
# MAGIC **Checkpoint**
# MAGIC
# MAGIC Mantiene el estado del procesamiento incremental y permite
# MAGIC que Auto Loader recuerde qué archivos ya procesó.
# MAGIC
# MAGIC **Record checksum**
# MAGIC
# MAGIC Identifica el contenido de un registro fuente y permite
# MAGIC comparar integridad y duplicidad.
# MAGIC
# MAGIC **Schema hash**
# MAGIC
# MAGIC Identifica la estructura del dataset utilizada durante
# MAGIC la carga.


# COMMAND ----------

# MAGIC %md
# MAGIC # 22. Definition of Done — E7.1
# MAGIC
# MAGIC - [ ] `brz_business` creada.
# MAGIC - [ ] `brz_user` creada.
# MAGIC - [ ] `brz_tip` creada.
# MAGIC - [ ] `brz_checkin` creada.
# MAGIC - [ ] `brz_review` creada.
# MAGIC - [ ] Datos RAW preservados sin transformación de negocio.
# MAGIC - [ ] Metadata técnica incorporada.
# MAGIC - [ ] `_record_checksum` generado.
# MAGIC - [ ] `_schema_hash` generado.
# MAGIC - [ ] Reconciliación RAW → Bronze validada para Batch.
# MAGIC - [ ] Auto Loader operativo para Review.
# MAGIC - [ ] Checkpoint persistente probado.
# MAGIC - [ ] Schema tracking habilitado.
# MAGIC - [ ] Experimento incremental documentado.
# MAGIC - [ ] Decisión Batch vs Auto Loader justificada.
# MAGIC - [ ] Evidencia de linaje capturada.


# COMMAND ----------

# ============================================================
# 23. RESULTADO FINAL
# ============================================================

execution_finished_at = (
    datetime
    .now(timezone.utc)
    .isoformat()
)


print()
print("=" * 70)
print(
    "PIPELINE RAW -> BRONZE FINALIZADO"
)
print("=" * 70)

print(
    f"Pipeline       : "
    f"{PIPELINE_NAME}"
)

print(
    f"Execution ID   : "
    f"{execution_id}"
)

print(
    f"Inicio UTC     : "
    f"{execution_started_at}"
)

print(
    f"Fin UTC        : "
    f"{execution_finished_at}"
)

print(
    f"Schema destino : "
    f"{CATALOG}."
    f"{BRONZE_SCHEMA}"
)

print()
print(
    "Entidades Batch:"
)

for (
    entity,
    raw_records,
    bronze_records,
    corrupt_records
) in batch_results:

    print(
        f"  ✔ {entity:<10} "
        f"RAW={raw_records:,} "
        f"BRONZE={bronze_records:,} "
        f"CORRUPTOS={corrupt_records:,}"
    )


print()
print(
    f"Review Auto Loader : "
    f"{review_count:,} filas totales"
)

print("=" * 70)